# Mesa de Ayuda IA — Patito S.A.
### Sistema Multi-Agente con LangChain + Google Gemini

**Semillero de Inteligencia Artificial — Universidad de Guayaquil**

Este notebook implementa una mesa de ayuda inteligente para el equipo de Marketing de Patito S.A. con **5 agentes especializados**:

| # | Agente | Función |
|---|--------|---------|
| 1 | Marca y Lineamientos | Logo, colores, tipografías, tono de voz |
| 2 | Campañas y Performance | Planificación, canales, KPIs |
| 3 | Cumplimiento Publicitario | Claims, opt-in, RGPD, influencers |
| 4 | Multimodal de Imagen | Análisis visual con Gemini Vision |
| 5 | Acción / Registro | Validar y registrar solicitudes de campaña |

**Stack:** Python · LangChain 0.3 · Google Gemini 2.0 Flash · ChromaDB · Pydantic

---
## Sección 1 — Instalación de dependencias

In [ ]:
%pip install \
    langchain-google-genai \
    langchain \
    langchain-community \
    langchain-chroma \
    chromadb \
    google-generativeai \
    python-dotenv \
    Pillow

print("✅ Dependencias instaladas.")


---
## Sección 2 — Configuración

> **Importante:** ingresa tu API Key de Google Gemini en la celda siguiente.
> Obtenla gratis en [aistudio.google.com](https://aistudio.google.com/)

In [ ]:
import os
import logging
from pathlib import Path

# ── Configuración de logging ──────────────────────────────────────────
logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

# ── API Key ───────────────────────────────────────────────────────────
# Opción A: variable de entorno (recomendado en producción)
# os.environ["GOOGLE_API_KEY"] = "AIzaSy..."

# Opción B: ingresar directamente (solo para pruebas)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = input("🔑 Ingresa tu Google API Key: ").strip()
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# ── Parámetros del sistema ────────────────────────────────────────────
GEMINI_MODEL           = "gemini-2.5-flash"
GEMINI_EMBEDDING_MODEL = "gemini-embedding-001"  # modelo anterior (text-embedding-004) fue retirado por Google el 14-ene-2026
CHROMA_PERSIST_DIR     = "./chroma_db"
CAMPAIGNS_FILE         = "./registro_campanas.txt"

CHUNK_SIZE      = 400
CHUNK_OVERLAP   = 80
TOP_K_RETRIEVAL = 4

# Nombres de colecciones Chroma (una por agente)
COLLECTION_MARCA       = "marca_lineamientos"
COLLECTION_CAMPANAS    = "campanas_kpis"
COLLECTION_CUMPLIMIENTO = "cumplimiento_publicitario"

print(f"✅ Configuración lista. Modelo: {GEMINI_MODEL}")

---
## Sección 3 — Base de conocimiento

Los tres documentos fuente se escriben directamente en disco para que el vector store los indexe.

In [ ]:
# Escribir los documentos de la base de conocimiento en disco
KB_FILES = {
    "01_Manual_de_Marca.txt":          'PATITO S.A.\nMANUAL DE MARCA Y LINEAMIENTOS VISUALES\n(Documento ficticio para fines de evaluación del semillero)\nBase de conocimiento del Agente de Marca y Lineamientos\n\n1. IDENTIDAD DE MARCA\nPatito S.A. es una empresa de productos tecnológicos de consumo. Su personalidad de marca\nes cercana, confiable y optimista. Toda comunicación debe transmitir simplicidad y calidez.\n\n2. LOGOTIPO\n2.1 Versiones oficiales: existen tres versiones del logotipo: a color (full color),\nmonocromático negro y monocromático blanco (negativo).\n2.2 Uso sobre fondos:\n- Sobre fondo blanco o claro: usar el logotipo a color.\n- Sobre fondos de color de la paleta oficial: se permite el logotipo a color SOLO si el\n  contraste es suficiente; si el fondo es oscuro o saturado, debe usarse la versión blanca\n  (negativo).\n- Sobre fotografías o fondos oscuros: usar siempre la versión blanca.\n2.3 Área de protección: mantener alrededor del logo un margen libre equivalente a la altura\nde la letra "P" del logotipo.\n2.4 Tamaño mínimo: 24 px de alto en digital y 15 mm en impreso.\n2.5 Usos incorrectos (prohibidos): deformar o estirar el logo, rotarlo, cambiar sus colores,\naplicarle sombras o efectos, o colocarlo sobre fondos que comprometan su legibilidad.\n\n3. PALETA DE COLORES\n3.1 Colores primarios:\n- Amarillo Patito: #FFC400\n- Azul Patito: #1F4E79\n3.2 Colores secundarios (apoyo): gris #555555, blanco #FFFFFF, verde menta #6FCF97.\n3.3 Regla clave: la paleta de marca NO debe modificarse para campañas estacionales. No se\npermite "ajustar" o crear nuevos colores primarios para una campaña de verano u otra\ntemporada. Para dar un aire estacional debe usarse la paleta secundaria de apoyo o\nfotografía, manteniendo intactos los colores primarios y el logotipo.\n\n4. TIPOGRAFÍAS\n- Tipografía principal: Montserrat (titulares).\n- Tipografía secundaria: Open Sans (cuerpo de texto).\n- En documentos internos de oficina se acepta Arial como alternativa.\n\n5. TONO DE VOZ\nClaro, positivo y respetuoso. Evitar tecnicismos innecesarios, mayúsculas sostenidas y\nlenguaje exagerado o sensacionalista.\n\n6. PLANTILLAS OFICIALES\nLas piezas deben construirse sobre las plantillas aprobadas por Marketing (redes sociales,\npresentaciones, email). Cualquier excepción requiere aprobación del área de Marca.',
    "02_Guia_Campanas_KPIs.txt":       'PATITO S.A.\nGUÍA DE CAMPAÑAS, CANALES Y KPIs DE MARKETING\n(Documento ficticio para fines de evaluación del semillero)\nBase de conocimiento del Agente de Campañas y Performance\n\n1. TIPOS DE CAMPAÑA SEGÚN OBJETIVO\n- Awareness (reconocimiento de marca).\n- Generación de leads (captación de contactos).\n- Conversión / ventas.\n- Fidelización / retención.\n\n2. CANALES DISPONIBLES\nRedes sociales (Meta, Instagram, TikTok, LinkedIn), Google Ads (search y display),\nemail marketing, sitio web/landing pages y marketing de contenidos.\n\n3. PROCESO DE UNA CAMPAÑA\n3.1 Brief y objetivo medible.  3.2 Definición de público y presupuesto.\n3.3 Producción de piezas (según Manual de Marca).  3.4 Lanzamiento.\n3.5 Monitoreo.  3.6 Reporte de cierre con KPIs.\n\n4. KPIs POR OBJETIVO\n4.1 Awareness: alcance, impresiones, frecuencia, CPM.\n4.2 Generación de leads (KPIs de cierre obligatorios):\n- Número de leads generados.\n- Costo por lead (CPL).\n- Tasa de conversión de visita a lead.\n- Tasa de conversión de lead a MQL (lead calificado por marketing).\n- CTR (tasa de clics).\n- Inversión total y CPL por canal.\n- ROI / ROAS de la campaña.\n4.3 Conversión/ventas: número de ventas, costo por adquisición (CPA), ticket promedio, ROAS.\n\n5. REPORTE DE CIERRE\nTodo cierre de campaña debe incluir: objetivo planteado vs. resultado, los KPIs del objetivo\ncorrespondiente, comparación contra la meta, aprendizajes y recomendaciones. Para campañas de\ngeneración de leads en redes sociales se reportan como mínimo: leads generados, CPL, tasa de\nconversión a lead y a MQL, CTR y ROAS.',
    "03_Cumplimiento_Publicitario.txt": 'PATITO S.A.\nLINEAMIENTOS DE CUMPLIMIENTO PUBLICITARIO Y PROTECCIÓN DE DATOS EN CAMPAÑAS\n(Documento ficticio para fines de evaluación del semillero)\nBase de conocimiento del Agente de Cumplimiento Publicitario\n\n1. PUBLICIDAD RESPONSABLE\nLa publicidad debe ser veraz y no engañosa. Las afirmaciones (claims) deben poder sustentarse.\n\n2. CLAIMS\n- Permitidos: beneficios reales y comprobables del producto.\n- Prohibidos: afirmaciones falsas, comparaciones engañosas, claims de salud no respaldados,\n  uso de "garantizado" o "el mejor" sin sustento, y precios o promociones que induzcan a error.\n\n3. CONSENTIMIENTO DE MARKETING (EMAIL Y MENSAJERÍA)\n- Solo se puede enviar comunicación comercial (email, SMS, WhatsApp) a personas que otorgaron\n  consentimiento explícito (opt-in) para recibir marketing.\n- NO está permitido enviar campañas de email a clientes que no dieron su consentimiento de\n  marketing, aunque sean clientes activos.\n- Toda comunicación debe incluir un mecanismo de baja (opt-out / "darse de baja") visible.\n- Las bajas deben procesarse de inmediato y respetarse.\n\n4. USO DE DATOS DE CLIENTES\n- Los datos solo se usan para la finalidad para la que fueron recolectados.\n- No se comparten datos con terceros sin base legal y sin consentimiento.\n- La segmentación debe respetar el consentimiento otorgado por cada contacto.\n\n5. INFLUENCERS Y CONTENIDO PAGADO\nTodo contenido pagado o con influencers debe identificarse claramente como publicidad\n(por ejemplo, #publicidad o "contenido pagado").\n\n6. ANTES DE LANZAR\nToda campaña debe validar: veracidad de claims, base de consentimiento del público objetivo,\ninclusión de opción de baja y cumplimiento del Manual de Marca.',
}

for filename, content in KB_FILES.items():
    Path(filename).write_text(content, encoding="utf-8")
    print(f"📄 Creado: {filename}")

print("\n✅ Base de conocimiento lista.")

---
## Sección 4 — Servicio de embeddings y vector stores

Cada agente RAG tiene su propia colección en ChromaDB. Los índices se persisten en disco para no regenerarlos en cada ejecución.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

def get_embeddings():
    """Instancia el modelo de embeddings de Gemini."""
    return GoogleGenerativeAIEmbeddings(
        model=GEMINI_EMBEDDING_MODEL,
        google_api_key=GOOGLE_API_KEY,
    )

def build_vector_store(txt_filename: str, collection_name: str, force_rebuild: bool = False) -> Chroma:
    """
    Carga un .txt, lo divide en chunks y construye (o carga) el vector store Chroma.
    Si ya existe en disco y force_rebuild=False, lo carga directamente.
    """
    persist_path = Path(CHROMA_PERSIST_DIR) / collection_name

    if persist_path.exists() and not force_rebuild:
        print(f" Cargando índice existente: {collection_name}")
        return Chroma(
            collection_name=collection_name,
            persist_directory=str(persist_path),
            embedding_function=get_embeddings(),
        )

    print(f"  ⏳ Construyendo índice: {collection_name}")
    loader = TextLoader(txt_filename, encoding="utf-8")
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    print(f"     {len(chunks)} chunks generados")

    persist_path.mkdir(parents=True, exist_ok=True)
    vs = Chroma.from_documents(
        documents=chunks,
        embedding=get_embeddings(),
        collection_name=collection_name,
        persist_directory=str(persist_path),
    )
    return vs

def get_retriever(txt_filename: str, collection_name: str, force_rebuild: bool = False):
    """Devuelve un retriever listo para usar en un agente RAG."""
    vs = build_vector_store(txt_filename, collection_name, force_rebuild)
    return vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K_RETRIEVAL})

print("✅ Funciones de embeddings definidas.")

---
## Sección 5 — Clase base para agentes RAG

In [ ]:
from dataclasses import dataclass, field
from typing import Any
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

NO_INFO_RESPONSE = "No encontré información suficiente en la base documental proporcionada."

RAG_PROMPT = """\
Eres un asistente especializado de Patito S.A. Tu área de expertise es: {area}.

INSTRUCCIONES:
- Responde ÚNICAMENTE usando la información del contexto documental proporcionado.
- Si la información no está en el contexto, responde exactamente:
  "No encontré información suficiente en la base documental proporcionada."
- No inventes, no supongas, no uses conocimiento general.
- Sé claro, directo y conciso. Usa el tono de la marca: claro, positivo y respetuoso.

CONTEXTO DOCUMENTAL:
{context}

PREGUNTA:
{question}

RESPUESTA:"""


@dataclass
class AgentResponse:
    agent_name: str
    answer: str
    sources: list[str] = field(default_factory=list)
    chunks_used: list[str] = field(default_factory=list)
    found_info: bool = True

    def to_dict(self) -> dict[str, Any]:
        return {
            "agent": self.agent_name,
            "answer": self.answer,
            "sources": self.sources,
            "found_info": self.found_info,
        }

    def display(self):
        sep = "─" * 60
        print(f"\n{sep}")
        print(f"🤖  {self.agent_name}")
        print(sep)
        print(self.answer)
        if self.sources:
            print(f"\n📄 Fuente: {', '.join(self.sources)}")
        print(sep)


class BaseRAGAgent:
    """Agente RAG base. Subclases declaran name, area y source_file."""
    name: str = "base"
    area: str = "información general"
    source_file: str = "documento"

    def __init__(self, retriever) -> None:
        self._retriever = retriever
        self._llm = ChatGoogleGenerativeAI(
            model=GEMINI_MODEL,
            google_api_key=GOOGLE_API_KEY,
            temperature=0.1,
        )
        prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
        self._chain = (
            {
                "context": self._retriever | (lambda docs: "\n\n---\n\n".join(d.page_content for d in docs)),
                "question": RunnablePassthrough(),
                "area": lambda _: self.area,
            }
            | prompt
            | self._llm
            | StrOutputParser()
        )

    def query(self, question: str) -> AgentResponse:
        docs = self._retriever.invoke(question)
        chunks = [d.page_content[:200] + "..." for d in docs]
        try:
            answer = self._chain.invoke(question)
        except Exception as e:
            return AgentResponse(agent_name=self.name, answer=f"Error: {e}", found_info=False)
        return AgentResponse(
            agent_name=self.name,
            answer=answer,
            sources=[self.source_file],
            chunks_used=chunks,
            found_info=NO_INFO_RESPONSE not in answer,
        )

print("✅ BaseRAGAgent y AgentResponse definidos.")

---
## Sección 6 — Agentes RAG especializados (3 agentes)

In [ ]:
# ── Agente 1: Marca y Lineamientos ───────────────────────────────────────────
class MarcaAgent(BaseRAGAgent):
    name        = "Agente de Marca y Lineamientos"
    area        = "identidad de marca, logotipo, paleta de colores, tipografías y tono de voz de Patito S.A."
    source_file = "01_Manual_de_Marca.txt"

# ── Agente 2: Campañas y Performance ────────────────────────────────────────
class CampanasAgent(BaseRAGAgent):
    name        = "Agente de Campañas y Performance"
    area        = "planificación de campañas, canales, KPIs y métricas de desempeño de Patito S.A."
    source_file = "02_Guia_Campanas_KPIs.txt"

# ── Agente 3: Cumplimiento Publicitario ──────────────────────────────────────
class CumplimientoAgent(BaseRAGAgent):
    name        = "Agente de Cumplimiento Publicitario"
    area        = "cumplimiento publicitario, protección de datos, consentimiento y claims de Patito S.A."
    source_file = "03_Cumplimiento_Publicitario.txt"

print("✅ Clases de agentes RAG definidas.")

---
## Sección 7 — Agente Multimodal de Imagen (Gemini Vision)

Analiza piezas gráficas y las contrasta con el Manual de Marca para detectar incumplimientos visuales.

In [ ]:
import base64
from langchain_core.messages import HumanMessage

MULTIMODAL_PROMPT = """\
Eres el Agente Multimodal de Imagen de Patito S.A.
Analiza piezas gráficas y verifica su cumplimiento con el Manual de Marca.

INSTRUCCIONES:
1. Describe brevemente el contenido visual de la imagen.
2. Identifica los elementos de marca: logotipo, colores, tipografías.
3. Compara con el Manual de Marca (contexto abajo).
4. Indica incumplimientos específicos si los hay.
5. Da un veredicto final: CUMPLE / NO CUMPLE / CUMPLE PARCIALMENTE.

CONTEXTO — MANUAL DE MARCA:
{marca_context}

Analiza la imagen y responde siguiendo las instrucciones."""


class MultimodalAgent:
    name = "Agente Multimodal de Imagen"

    def __init__(self, marca_retriever) -> None:
        self._llm = ChatGoogleGenerativeAI(
            model=GEMINI_MODEL, google_api_key=GOOGLE_API_KEY, temperature=0.1
        )
        self._marca_retriever = marca_retriever

    def analyze_image(self, image_path: str, extra_question: str = "") -> AgentResponse:
        """Analiza una imagen local y la contrasta con el Manual de Marca."""
        # Recuperar contexto del Manual de Marca
        query = extra_question or "logotipo colores tipografía paleta marca"
        docs = self._marca_retriever.invoke(query)
        marca_context = "\n\n".join(d.page_content for d in docs)

        # Codificar imagen en base64
        path = Path(image_path)
        ext_to_mime = {".jpg": "image/jpeg", ".jpeg": "image/jpeg",
                       ".png": "image/png", ".webp": "image/webp", ".gif": "image/gif"}
        media_type = ext_to_mime.get(path.suffix.lower(), "image/jpeg")
        img_b64 = base64.b64encode(path.read_bytes()).decode()

        # Construir mensaje multimodal
        system_text = MULTIMODAL_PROMPT.format(marca_context=marca_context)
        user_text = extra_question or "Analiza esta pieza y verifica cumplimiento con el Manual de Marca de Patito S.A."

        message = HumanMessage(content=[
            {"type": "text", "text": system_text + "\n\n" + user_text},
            {"type": "image_url", "image_url": {"url": f"data:{media_type};base64,{img_b64}"}},
        ])

        try:
            response = self._llm.invoke([message])
            answer = response.content
        except Exception as e:
            return AgentResponse(agent_name=self.name, answer=f"Error al analizar imagen: {e}", found_info=False)

        return AgentResponse(
            agent_name=self.name,
            answer=answer,
            sources=["01_Manual_de_Marca.txt", path.name],
            found_info=True,
        )

    def analyze_image_bytes(self, img_bytes: bytes, media_type: str = "image/jpeg",
                             extra_question: str = "") -> AgentResponse:
        """Analiza imagen a partir de bytes (para uso programático)."""
        import tempfile, os
        ext = {"image/jpeg": ".jpg", "image/png": ".png",
                "image/webp": ".webp", "image/gif": ".gif"}.get(media_type, ".jpg")
        with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
            tmp.write(img_bytes); tmp_path = tmp.name
        try:
            return self.analyze_image(tmp_path, extra_question)
        finally:
            os.unlink(tmp_path)

print("✅ MultimodalAgent definido.")

---
## Sección 8 — Agente de Acción (Registro de Campañas)

Extrae datos de campaña desde lenguaje natural, valida campos obligatorios, pide confirmación y registra en un archivo `.txt`.

In [ ]:
import json, uuid
from datetime import datetime

REQUIRED_FIELDS = ["nombre", "objetivo", "canales", "publico_objetivo",
                   "presupuesto", "fecha_inicio", "fecha_fin"]
EMAIL_CHANNELS  = {"email", "correo", "e-mail", "sms", "whatsapp", "mensajería"}

FIELD_LABELS = {
    "nombre":                  "nombre de la campaña",
    "objetivo":                "objetivo (awareness / leads / conversión / fidelización)",
    "canales":                 "canal(es) de campaña",
    "publico_objetivo":        "público objetivo",
    "presupuesto":             "presupuesto (monto y moneda)",
    "fecha_inicio":            "fecha de inicio",
    "fecha_fin":               "fecha de fin",
    "consentimiento_marketing":"confirmación de que el público dio consentimiento opt-in para ese canal",
}

EXTRACT_PROMPT = """\
Extrae los campos de la solicitud de campaña del siguiente texto.
Devuelve ÚNICAMENTE un JSON con los campos encontrados (omite los ausentes, no pongas null).

Campos: nombre, objetivo, canales (lista), publico_objetivo, presupuesto, fecha_inicio,
fecha_fin, consentimiento_marketing (true/false, solo si canal es email/sms/whatsapp).

TEXTO:
{text}

JSON (solo el objeto, sin markdown):"""

CONFIRM_PROMPT = """\
Genera un resumen amigable de la campaña para solicitar confirmación al usuario.
Incluye todos los datos en formato legible.
Termina con: ¿Confirmas el registro? (responde sí o no)

Datos:
{data}"""


@dataclass
class CampaignData:
    nombre: str = ""
    objetivo: str = ""
    canales: list = field(default_factory=list)
    publico_objetivo: str = ""
    presupuesto: str = ""
    fecha_inicio: str = ""
    fecha_fin: str = ""
    consentimiento_marketing: Any = None

    def requires_consent(self) -> bool:
        return bool({c.lower() for c in self.canales} & EMAIL_CHANNELS)

    def missing_fields(self) -> list[str]:
        missing = [f for f in REQUIRED_FIELDS if not getattr(self, f, None)]
        if self.requires_consent() and self.consentimiento_marketing is None:
            missing.append("consentimiento_marketing")
        return missing

    def to_dict(self):
        return {
            "nombre": self.nombre, "objetivo": self.objetivo,
            "canales": self.canales, "publico_objetivo": self.publico_objetivo,
            "presupuesto": self.presupuesto, "fecha_inicio": self.fecha_inicio,
            "fecha_fin": self.fecha_fin, "consentimiento_marketing": self.consentimiento_marketing,
        }


class AccionAgent:
    name = "Agente de Acción (Registro)"

    def __init__(self) -> None:
        self._llm = ChatGoogleGenerativeAI(
            model=GEMINI_MODEL, google_api_key=GOOGLE_API_KEY, temperature=0
        )
        self._register_file = Path(CAMPAIGNS_FILE)
        self._pending: CampaignData | None = None

    def _extract(self, text: str) -> CampaignData:
        prompt = ChatPromptTemplate.from_template(EXTRACT_PROMPT)
        chain = prompt | self._llm | StrOutputParser()
        try:
            raw = chain.invoke({"text": text}).strip().strip("```json").strip("```").strip()
            data = json.loads(raw)
        except Exception:
            data = {}
        cd = CampaignData()
        cd.nombre            = data.get("nombre", "")
        cd.objetivo          = data.get("objetivo", "")
        canales              = data.get("canales", [])
        cd.canales           = canales if isinstance(canales, list) else [canales]
        cd.publico_objetivo  = data.get("publico_objetivo", "")
        cd.presupuesto       = data.get("presupuesto", "")
        cd.fecha_inicio      = data.get("fecha_inicio", "")
        cd.fecha_fin         = data.get("fecha_fin", "")
        consent              = data.get("consentimiento_marketing")
        if isinstance(consent, bool):   cd.consentimiento_marketing = consent
        elif isinstance(consent, str):  cd.consentimiento_marketing = consent.lower() in ("true","sí","si","yes")
        return cd

    def _write(self, cd: CampaignData) -> str:
        reg_id = str(uuid.uuid4())[:8].upper()
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        consent_line = (f"\nConsentimiento: {'SÍ' if cd.consentimiento_marketing else 'NO'}"
                        if cd.requires_consent() else "")
        entry = (
            f"\n{'='*55}\n"
            f"ID: {reg_id} | Registrado: {ts}\n"
            f"Nombre: {cd.nombre}\n"
            f"Objetivo: {cd.objetivo}\n"
            f"Canales: {', '.join(cd.canales)}\n"
            f"Público: {cd.publico_objetivo}\n"
            f"Presupuesto: {cd.presupuesto}\n"
            f"Período: {cd.fecha_inicio} → {cd.fecha_fin}"
            f"{consent_line}\n"
            f"{'='*55}\n"
        )
        with open(self._register_file, "a", encoding="utf-8") as f:
            f.write(entry)
        return reg_id

    def process_request(self, text: str) -> AgentResponse:
        cd = self._extract(text)
        missing = cd.missing_fields()
        if missing:
            faltantes = "\n".join(f"  • {FIELD_LABELS.get(f, f)}" for f in missing)
            return AgentResponse(
                agent_name=self.name,
                answer=f"Para registrar la campaña necesito los siguientes datos:\n\n{faltantes}\n\nPor favor complétalos.",
            )
        prompt = ChatPromptTemplate.from_template(CONFIRM_PROMPT)
        chain = prompt | self._llm | StrOutputParser()
        summary = chain.invoke({"data": str(cd.to_dict())})
        self._pending = cd
        return AgentResponse(agent_name=self.name, answer=summary, sources=["registro_campanas.txt"])

    def confirm(self) -> AgentResponse:
        if not self._pending:
            return AgentResponse(agent_name=self.name, answer="No hay registro pendiente de confirmación.", found_info=False)
        try:
            reg_id = self._write(self._pending)
            answer = (f"✅ Campaña registrada. ID: {reg_id}\n"
                      f"Guardado en: {self._register_file}")
        except Exception as e:
            answer = f"Error al guardar: {e}"
        self._pending = None
        return AgentResponse(agent_name=self.name, answer=answer, sources=["registro_campanas.txt"])

    def cancel(self) -> AgentResponse:
        self._pending = None
        return AgentResponse(agent_name=self.name, answer="Registro cancelado. No se guardó ningún dato.")

print("✅ AccionAgent definido.")

---
## Sección 9 — Orquestador principal

Clasifica la intención de cada consulta con Gemini y enruta al agente correcto automáticamente.

In [ ]:
CLASSIFIER_PROMPT = """\
Clasifica la siguiente consulta en el agente más adecuado del sistema de IA de Patito S.A.
Devuelve ÚNICAMENTE un JSON con esta estructura:
{{"agent": "<marca|campanas|cumplimiento|accion|multimodal|general>", "confidence": <0.0-1.0>, "reasoning": "<máx 20 palabras>"}}

Reglas:
- marca: logotipo, colores, paleta, tipografía, tono de voz, identidad visual.
- campanas: planificación de campañas, canales, KPIs, métricas, objetivos de marketing.
- cumplimiento: claims, opt-in, RGPD, datos, influencers, legalidad publicitaria.
- accion: el usuario quiere REGISTRAR, CREAR o GUARDAR una nueva campaña.
- multimodal: adjunta imagen o pide análisis visual de una pieza gráfica.
- general: saludos, fuera de scope, no encaja en ninguna categoría.

CONSULTA: {query}

JSON:"""

GENERAL_PROMPT = """\
Eres el asistente de la Mesa de Ayuda IA de Patito S.A.
Si el usuario saluda o pregunta algo fuera del scope, responde amablemente e indica que puedes ayudar con:
- Lineamientos de marca (logo, colores, tipografías)
- Planificación de campañas y KPIs
- Cumplimiento publicitario y protección de datos
- Análisis visual de piezas gráficas
- Registro de solicitudes de campaña

Consulta: {query}"""


@dataclass
class OrchestratorResponse:
    query: str
    agent_used: str
    agent_response: AgentResponse
    intent_confidence: float = 1.0
    intent_reasoning: str = ""

    def display(self):
        print(f"\n{'═'*60}")
        print(f"  📨 Consulta : {self.query[:80]}{'...' if len(self.query)>80 else ''}")
        print(f"  🎯 Agente   : {self.agent_used}")
        print(f"  📊 Confianza: {self.intent_confidence:.0%}  |  {self.intent_reasoning}")
        print(f"{'═'*60}")
        self.agent_response.display()


class Orchestrator:
    """Orquestador principal — instancia todos los agentes una sola vez."""

    def __init__(self, force_rebuild: bool = False) -> None:
        print("⏳ Iniciando agentes...")
        self._llm = ChatGoogleGenerativeAI(
            model=GEMINI_MODEL, google_api_key=GOOGLE_API_KEY, temperature=0
        )
        # Retrievers (uno por colección Chroma)
        ret_marca       = get_retriever("01_Manual_de_Marca.txt",          COLLECTION_MARCA,        force_rebuild)
        ret_campanas    = get_retriever("02_Guia_Campanas_KPIs.txt",       COLLECTION_CAMPANAS,     force_rebuild)
        ret_cumplimiento = get_retriever("03_Cumplimiento_Publicitario.txt", COLLECTION_CUMPLIMIENTO, force_rebuild)

        # Instanciar agentes
        self.marca        = MarcaAgent(ret_marca)
        self.campanas     = CampanasAgent(ret_campanas)
        self.cumplimiento = CumplimientoAgent(ret_cumplimiento)
        self.multimodal   = MultimodalAgent(ret_marca)
        self.accion       = AccionAgent()

        self._awaiting_confirmation = False
        print("✅ Orquestador listo. Todos los agentes activos.")

    def _classify(self, query: str) -> dict:
        prompt = ChatPromptTemplate.from_template(CLASSIFIER_PROMPT)
        chain = prompt | self._llm | StrOutputParser()
        try:
            raw = chain.invoke({"query": query}).strip().strip("```json").strip("```").strip()
            return json.loads(raw)
        except Exception:
            return {"agent": "general", "confidence": 0.5, "reasoning": "Error de clasificación."}

    def _general(self, query: str) -> AgentResponse:
        prompt = ChatPromptTemplate.from_template(GENERAL_PROMPT)
        chain = prompt | self._llm | StrOutputParser()
        return AgentResponse(agent_name="Asistente General", answer=chain.invoke({"query": query}))

    def process(self, query: str, image_path: str | None = None) -> OrchestratorResponse:
        # ── Flujo de confirmación del agente de acción ────────────────────────
        if self._awaiting_confirmation:
            q = query.strip().lower()
            if q in ("sí","si","yes","confirmo","ok","1"):
                resp = self.accion.confirm()
                self._awaiting_confirmation = False
            elif q in ("no","cancelar","cancel","2"):
                resp = self.accion.cancel()
                self._awaiting_confirmation = False
            else:
                resp = AgentResponse(
                    agent_name=self.accion.name,
                    answer="No entendí. Responde **sí** para confirmar o **no** para cancelar.",
                )
            return OrchestratorResponse(query=query, agent_used=resp.agent_name,
                                        agent_response=resp, intent_confidence=1.0,
                                        intent_reasoning="Confirmación pendiente.")

        # ── Imagen adjunta → multimodal ───────────────────────────────────────
        if image_path:
            resp = self.multimodal.analyze_image(image_path, extra_question=query)
            return OrchestratorResponse(query=query, agent_used=resp.agent_name,
                                        agent_response=resp, intent_confidence=1.0,
                                        intent_reasoning="Imagen adjunta → agente multimodal.")

        # ── Clasificar y enrutar ──────────────────────────────────────────────
        intent = self._classify(query)
        key        = intent.get("agent", "general")
        confidence = float(intent.get("confidence", 0.5))
        reasoning  = intent.get("reasoning", "")

        if key == "marca":
            resp = self.marca.query(query)
        elif key == "campanas":
            resp = self.campanas.query(query)
        elif key == "cumplimiento":
            resp = self.cumplimiento.query(query)
        elif key == "accion":
            resp = self.accion.process_request(query)
            if "¿Confirmas el registro?" in resp.answer or "sí o no" in resp.answer.lower():
                self._awaiting_confirmation = True
        elif key == "multimodal":
            resp = AgentResponse(
                agent_name=self.multimodal.name,
                answer="Para analizar una pieza gráfica pasa la ruta de la imagen en el parámetro `image_path`.",
            )
        else:
            resp = self._general(query)

        return OrchestratorResponse(query=query, agent_used=resp.agent_name,
                                    agent_response=resp, intent_confidence=confidence,
                                    intent_reasoning=reasoning)

print("✅ Orchestrator definido.")

---
##  Sección 10 — Inicialización del sistema

> Ejecuta esta celda una sola vez. Los índices vectoriales se guardan en `./chroma_db/` y se reutilizan en ejecuciones posteriores.

In [ ]:
# Construye los vector stores (si ya existen en disco, los carga directamente)
orquestador = Orchestrator(force_rebuild=False)

---
##  Sección 11 — Demos por agente

Cada celda demuestra un agente específico con una consulta real.

###  Demo 1 — Agente de Marca y Lineamientos

In [ ]:
resultado = orquestador.process("¿Cuáles son los colores primarios de Patito S.A. y en qué casos se permite usar el logotipo blanco?")
resultado.display()

###  Demo 2 — Agente de Campañas y Performance

In [ ]:
resultado = orquestador.process("¿Qué KPIs son obligatorios en el reporte de cierre de una campaña de generación de leads?")
resultado.display()

###  Demo 3 — Agente de Cumplimiento Publicitario

In [ ]:
resultado = orquestador.process("¿Puedo usar el claim 'el mejor precio garantizado' en un anuncio de Google Ads?")
resultado.display()

###  Demo 4 — Agente de Acción (Registro de Campaña)

Flujo completo: solicitud con datos incompletos → datos completos → confirmación → guardado.

In [ ]:
# Paso 1: solicitud incompleta (faltan datos)
r1 = orquestador.process("Quiero registrar una campaña de Instagram")
r1.display()

In [ ]:
# Paso 2: solicitud con todos los datos
r2 = orquestador.process(
    "Quiero registrar la campaña 'Verano Patito' en Instagram y Facebook, "
    "objetivo conversión, público jóvenes 18-30 años, presupuesto USD 800, "
    "del 1 al 31 de julio"
)
r2.display()

In [ ]:
# Paso 3: confirmar el registro
r3 = orquestador.process("sí")
r3.display()

### 🖼️ Demo 5 — Agente Multimodal de Imagen

Para analizar una imagen real, sube un archivo al entorno y usa `image_path`.

In [ ]:
# Ejemplo con imagen real (descomenta y ajusta la ruta):
# resultado = orquestador.process(
#     query="¿Los colores y tipografías cumplen con el Manual de Marca?",
#     image_path="./mi_banner.png"
# )
# resultado.display()

# ── Demo simulado sin imagen ──────────────────────────────────────────────────
resultado = orquestador.process("Analiza esta pieza gráfica")   # sin image_path
resultado.display()

---
## 💬 Sección 12 — Consultas libres

Usa estas celdas para hacer tus propias consultas. El orquestador detecta el agente automáticamente.

In [ ]:
# ── Modifica esta celda con tu consulta ──────────────────────────────────────
MI_CONSULTA = "¿Puedo usar una tipografía diferente a Montserrat si es una campaña especial?"

resultado = orquestador.process(MI_CONSULTA)
resultado.display()

In [ ]:
# ── Otra consulta de ejemplo ─────────────────────────────────────────────────
MI_CONSULTA_2 = "¿Qué debo incluir en el reporte de cierre de una campaña de awareness?"

resultado2 = orquestador.process(MI_CONSULTA_2)
resultado2.display()

---
## 📋 Sección 13 — Ver campañas registradas

In [ ]:
reg_file = Path(CAMPAIGNS_FILE)
if reg_file.exists() and reg_file.stat().st_size > 0:
    print(reg_file.read_text(encoding="utf-8"))
else:
    print("No hay campañas registradas aún.")

---
## 🔄 Sección 14 — Reconstruir índices vectoriales

Ejecuta esta celda solo si modificaste algún documento de la base de conocimiento.

In [ ]:
# ADVERTENCIA: esto regenera los embeddings (tarda ~10-20 segundos y consume tokens)
# orquestador = Orchestrator(force_rebuild=True)
# print("✅ Índices reconstruidos.")
print("ℹ️  Descomenta las líneas anteriores y ejecuta para reconstruir.")